# Setup

In [1]:
! pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.9 MB/s eta 0:00:00


Biblioteka Cohere to narzędzie do pracy z modelami językowymi. Umożliwia generowanie tekstu, embeddingi (przekształcanie tekstu na wartości liczbowe), klasyfikowanie treści i inne zadania związane z przetwarzaniem języka naturalnego.

In [ ]:
import cohere
from google.colab import userdata
import os

`import cohere` Ta biblioteka umożliwia komunikację z modelami językowymi Cohere poprzez ich API.

In [ ]:
api_key = userdata.get("cohereprod")
os.environ["COHERE_API_KEY"] = api_key

co = cohere.Client(api_key)

In [4]:
class CFG:
    model = "command-r-plus"

W tym przypadku, klasa zawiera tylko jedną zmienną (atrybut):
- `model = "command-r-plus"` - definiuje konkretny model Cohere, który będzie używany do przetwarzania języka naturalnego.

Model "command-r-plus" to jeden z zaawansowanych modeli oferowanych przez firmę Cohere. Jest to model instrukcyjny (command model), co oznacza, że został zaprojektowany do wykonywania konkretnych poleceń i zadań językowych.

# Funkcje

In [ ]:
def daily_sales_report(day: str) -> dict:
    """
    Function to retrieve the sales report for the given day
    """
    # Mock database containing daily sales reports
    sales_database = {
        "2023-09-28": {"total_sales_amount": 5000, "total_units_sold": 100},
        "2023-09-29": {"total_sales_amount": 10000, "total_units_sold": 250},
        "2023-09-30": {"total_sales_amount": 8000, "total_units_sold": 200},
    }

    report = sales_database.get(day, {})

    if report:
        return {
            "date": day,
            "summary": f"Total Sales Amount: {report['total_sales_amount']}, Total Units Sold: {report['total_units_sold']}",
        }
    else:
        return {"date": day, "summary": "No sales data available for this day."}


In [ ]:
def product_database(category: str) -> dict:
    """
    Function to retrieve products for the given category
    """

    # Mock product catalog
    product_catalog = {
        "Electronics": [
            {
                "product_id": "E1001",
                "name": "Smartphone",
                "price": 500,
                "stock_level": 20,
            },
            {"product_id": "E1002", "name": "Laptop", "price": 1000, "stock_level": 15},
            {"product_id": "E1003", "name": "Tablet", "price": 300, "stock_level": 25},
        ],
        "Clothing": [
            {"product_id": "C1001", "name": "T-Shirt", "price": 20, "stock_level": 100},
            {"product_id": "C1002", "name": "Jeans", "price": 50, "stock_level": 80},
            {"product_id": "C1003", "name": "Jacket", "price": 100, "stock_level": 40},
        ],
    }

    products = product_catalog.get(category, [])
    return {"category": category, "products": products}


functions_map = {
    "daily_sales_report": daily_sales_report,
    "product_database": product_database,
}

In [ ]:
tools = [
    {
        "name": "daily_sales_report",
        "description": "Connects to a database to retrieve overall sales volumes and sales information for a given day.",
        "parameter_definitions": {
            "day": {
                "description": "Retrieves sales data for this day, formatted as YYYY-MM-DD.",
                "type": "str",
                "required": True,
            }
        },
    },
    {
        "name": "product_database",
        "description": "A database that contains information about all the products of this company, including categories, prices, and stock levels.",
        "parameter_definitions": {
            "category": {
                "description": "Retrieves product information data for all products in this category.",
                "type": "str",
                "required": True,
            }
        },
    },
]

Ta część kodu definiuje listę `tools`, która zawiera opisy narzędzi dostępnych dla modelu Cohere. Narzędzia te umożliwiają modelowi dostęp do zewnętrznych źródeł danych, co znacznie rozszerza jego możliwości. Przyjrzyjmy się szczegółowo tej strukturze.

Lista `tools` zawiera dwa słowniki, każdy opisujący jedno narzędzie:

**Pierwsze narzędzie - `daily_sales_report`:**
Ten element umożliwia modelowi pobieranie raportów sprzedaży dla konkretnego dnia. Ma on następujące właściwości:
- `name`: identyfikator narzędzia, który musi odpowiadać nazwie funkcji w `functions_map`
- `description`: opis wyjaśniający modelowi, do czego służy to narzędzie (łączy się z bazą danych w celu pobrania informacji o sprzedaży)
- `parameter_definitions`: definicje parametrów, które muszą być przekazane do tego narzędzia
  - `day`: jedyny parametr tego narzędzia, który musi być w formacie "RRRR-MM-DD"
  - Parametr ten jest oznaczony jako wymagany (`required: True`) i ma określony typ (`type: "str"`)

**Drugie narzędzie - `product_database`:**
To narzędzie umożliwia modelowi dostęp do bazy danych produktów. Podobnie jak poprzednie, zawiera:
- `name`: identyfikator "product_database"
- `description`: wyjaśnienie, że ta funkcja daje dostęp do informacji o produktach, kategoriach, cenach i poziomach zapasów
- `parameter_definitions`: określa, że narzędzie to wymaga jednego parametru:
  - `category`: nazwa kategorii produktów, do której model chce uzyskać dostęp
  - Podobnie jak w przypadku pierwszego narzędzia, parametr ten jest wymagany i ma typ string

Ta struktura jest kluczowa dla funkcjonowania systemu z kilku powodów:

1. **Informacja dla modelu**: Opisy narzędzi i parametrów służą jako wskazówki dla modelu, pomagając mu zrozumieć, kiedy i jak używać danego narzędzia. Model decyduje, które narzędzie wywołać, opierając się na tych opisach.

2. **Walidacja parametrów**: Definicje parametrów określają, jakie dane model powinien przekazać do narzędzia. System może wykorzystać te definicje do sprawdzenia, czy model przekazuje poprawne wartości.

3. **Komunikacja interfejsu**: Ta struktura tworzy czytelny interfejs między modelem języka a zewnętrznymi funkcjami, standaryzując sposób, w jaki model wchodzi w interakcję z danymi i usługami.

W kontekście całego systemu, gdy użytkownik zadaje pytanie dotyczące sprzedaży lub produktów, model może zdecydować, że potrzebuje konkretnych danych, aby udzielić rzetelnej odpowiedzi. Używając tych narzędzi, może pobrać te dane, zanim udzieli ostatecznej odpowiedzi, co sprawia, że odpowiedzi są bardziej precyzyjne i oparte na faktach.

In [9]:
preamble = """## Task & Context
You are an assistant for an e-commerce company. You will be asked a very wide array of requests \
        on all kinds of topics. You will be equipped with a set of tools, which you use to get your answer. \
        You should focus on serving the user's needs as best you can, which will be wide-ranging.

## Style Guide
Unless the user asks for a different style of answer, you should answer in full sentences,\
    using proper grammar and spelling.
"""

Ten kod definiuje zmienną `preamble` zawierającą instrukcje dla modelu Cohere. Jest to istotny element w architekturze systemów wykorzystujących modele językowe, ponieważ kształtuje sposób, w jaki model będzie działał i odpowiadał.

Preambuła składa się z dwóch głównych sekcji:

**Task & Context (Zadanie i Kontekst):**
Ta sekcja definiuje rolę i środowisko działania modelu. Model zostaje poinformowany, że ma działać jako asystent dla firmy e-commerce. Jest to kluczowe dla ustawienia kontekstu - dzięki temu model wie, że pytania będą dotyczyć głównie tematów związanych z handlem elektronicznym. Jednocześnie model jest przygotowany na bardzo szeroką gamę zapytań z różnych dziedzin, co odzwierciedla rzeczywiste użytkowanie asystentów w biznesie.

Ważnym elementem tej sekcji jest też informacja o dostępnych narzędziach. Model jest instruowany, aby korzystał z tych narzędzi w celu uzyskania odpowiedzi, co bezpośrednio łączy się z wcześniej zdefiniowanymi funkcjami i mechanizmem wywoływania narzędzi w funkcji `run_assistant`.

**Style Guide (Przewodnik Stylistyczny):**
Ta sekcja określa, jak model ma formułować swoje odpowiedzi. Domyślnie, model powinien odpowiadać pełnymi zdaniami, używając poprawnej gramatyki i pisowni. Jest to standard komunikacji profesjonalnej i biznesowej. Jednocześnie jest tu zawarta elastyczność - jeśli użytkownik poprosi o inny styl odpowiedzi, model może dostosować swój sposób komunikacji.

Preambuła pełni kilka kluczowych funkcji w systemie:

1. **Ukierunkowanie modelu** - Model ma jasno określoną rolę i cel, co pomaga mu generować bardziej spójne i trafne odpowiedzi.

2. **Spójność komunikacji** - Określenie stylu odpowiedzi zapewnia jednolity ton i format we wszystkich interakcjach z użytkownikiem.

3. **Efektywność narzędzi** - Poinformowanie modelu o dostępnych narzędziach i zachęcenie go do ich używania zwiększa prawdopodobieństwo, że model będzie efektywnie korzystał z zewnętrznych źródeł danych.

4. **Adaptacyjność** - Model ma instrukcję, aby dostosować się do potrzeb użytkownika, co czyni go bardziej użytecznym w różnych scenariuszach.

W kontekście całego systemu, preambuła jest pierwszym krokiem w komunikacji z modelem, ustalając ramy i oczekiwania dla wszystkich dalszych interakcji. Jest to praktyka powszechnie stosowana w systemach opartych na modelach językowych, gdzie jasne określenie roli i oczekiwań znacząco wpływa na jakość generowanych odpowiedzi.

In [ ]:
def run_assistant(message, chat_history=None):
    if chat_history is None:
        chat_history = []

    # Step 1: Get user message
    print(f"Question:\n{message}")
    print("-" * 50)

    # Step 2: Generate tool calls (if any)
    response = co.chat(
        message=message,
        model=CFG.model,
        preamble=preamble,
        tools=tools,
        chat_history=chat_history,
        force_single_step=True,
    )

    while response.tool_calls:
        tool_calls = response.tool_calls

        if response.text:
            print("Tool plan:")
            print(response.text, "\n")
        print("Tool calls:")
        for call in tool_calls:
            print(f"Tool name: {call.name} | Parameters: {call.parameters}")
        print("=" * 50)

        # Step 3: Get tool results
        tool_results = []
        for tc in tool_calls:
            tool_call = {"name": tc.name, "parameters": tc.parameters}
            tool_output = functions_map[tc.name](**tc.parameters)
            tool_results.append({"call": tool_call, "outputs": [tool_output]})

        # Step 4: Generate response and citations
        response = co.chat(
            message="",
            model=CFG.model,
            preamble=preamble,
            tools=tools,
            tool_results=tool_results,
            chat_history=response.chat_history,
        )

        # Append the current chat turn to the chat history
        chat_history = response.chat_history

    # Print final response
    print("Final response:")
    print(response.text)
    print("=" * 50)

    # Print citations (if any)
    if response.citations:
        print("Citations:")
        for citation in response.citations:
            print(citation)
        print("\nCited Documents:")
        for document in response.documents:
            print(document)
        print("=" * 50)

    return chat_history

Ta funkcja `run_assistant` jest sercem systemu konwersacyjnego opartego na modelu Cohere. Jej zadaniem jest obsługa całego procesu komunikacji z modelem, włącznie z wykonywaniem narzędzi (tools) i zbieraniem ich wyników. Przeanalizujmy jej działanie krok po kroku:

Funkcja przyjmuje dwa parametry:
- `message` - wiadomość od użytkownika
- `chat_history` - historię dotychczasowej konwersacji (z wartością domyślną `None`)

Na początku funkcja sprawdza, czy historia konwersacji została przekazana. Jeśli nie, inicjalizuje ją jako pustą listę.

Następnie rozpoczyna się główny proces, który składa się z kilku etapów:

**Etap 1:** Wyświetlenie wiadomości użytkownika. Jest to prosta operacja użyta do celów diagnostycznych, aby widoczne było, na jakie pytanie odpowiada system.

**Etap 2:** Generowanie wywołań narzędzi. Tutaj wywołujemy metodę `co.chat()` klienta Cohere, przekazując:
- Wiadomość użytkownika
- Model języka (z konfiguracji `CFG.model`)
- Preamble (kontekst lub instrukcje dla modelu)
- Dostępne narzędzia (tools)
- Historię konwersacji
- Parametr `force_single_step=True`, który wymusza, aby model najpierw zaproponował plan działania z użyciem narzędzi, zanim wygeneruje ostateczną odpowiedź

**Etap 3 i 4 (w pętli):** Wykonywanie narzędzi i generowanie odpowiedzi. Ta część kodu działa w pętli `while`, która trwa dopóki model sugeruje użycie jakichś narzędzi:
1. Pobieramy wywołania narzędzi z odpowiedzi modelu
2. Wyświetlamy plan działania i szczegóły wywołań narzędzi (jeśli istnieją)
3. Dla każdego wywołania narzędzia:
   - Tworzymy słownik z nazwą narzędzia i parametrami
   - Wykonujemy odpowiednią funkcję z `functions_map`, przekazując parametry
   - Zbieramy wyniki w liście `tool_results`
4. Ponownie wywołujemy model, tym razem przekazując wyniki wykonania narzędzi
5. Aktualizujemy historię konwersacji

Po zakończeniu pętli (gdy model nie sugeruje już użycia narzędzi), funkcja wyświetla finalną odpowiedź oraz cytowania (jeśli istnieją).

Ta funkcja jest doskonałym przykładem implementacji tzw. "agentic AI" lub "AI z narzędziami" (tool-using AI):
1. Model analizuje zapytanie użytkownika i decyduje, jakich narzędzi potrzebuje do udzielenia odpowiedzi
2. System wykonuje te narzędzia, zbierając rzeczywiste dane
3. Model dostaje wyniki i generuje ostateczną odpowiedź, bazując na faktycznych danych

Taka architektura ma kilka kluczowych zalet:
- Model może korzystać z aktualnych danych spoza swojego treningu
- Może wykonywać konkretne operacje na danych, których nie mógłby wykonać samodzielnie
- Cały proces jest transparentny - widzimy, jakie narzędzia model chce użyć i jakie dane otrzymuje

Jest to nowoczesne podejście do budowania asystentów AI, które łączy generatywne możliwości modeli językowych z dostępem do zewnętrznych źródeł danych i funkcji.

# Model

## Single step

In [10]:
chat_history = run_assistant("Can you provide a sales summary for 29th September 2023?")

Question:
Can you provide a sales summary for 29th September 2023?
--------------------------------------------------
Tool calls:
Tool name: daily_sales_report | Parameters: {'day': '2023-09-29'}
Final response:
On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.
Citations:
start=30 end=39 text='250 units' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'
start=64 end=70 text='10,000' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'

Cited Documents:
{'date': '2023-09-29', 'id': 'daily_sales_report:0:2:0', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250', 'tool_name': 'daily_sales_report'}


In [ ]:
for turn in chat_history:
    print(turn, "\n")

role='USER' message='Can you provide a sales summary for 29th September 2023?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), outputs=[{'date': '2023-09-29', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250'}])] 

role='CHATBOT' message='On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.' tool_calls=None 

role='USER' message='What about the 28th?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'}), outputs=[{'date': '2023-09-28', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100'}])] 

role='CHATBOT' message='On 28 September 2023, we sold 100 units, totalling a

## Single step - parallel

In [ ]:
chat_history = run_assistant(
    "Can you provide a sales summary for 28th and 29th September 2023 \
                    as well as the stock level of the products in the 'Electronics' category?"
)

Question:
Can you provide a sales summary for 28th and 29th September 2023                     as well as the stock level of the products in the 'Electronics' category?
--------------------------------------------------
Tool calls:
Tool name: daily_sales_report | Parameters: {'day': '2023-09-28'}
Tool name: daily_sales_report | Parameters: {'day': '2023-09-29'}
Tool name: product_database | Parameters: {'category': 'Electronics'}
Final response:
On 28 September 2023, the total sales amount was 5000, with 100 units sold. The following day, 29 September 2023, the total sales amount was 10000, with 250 units sold. 

Here is the stock level of the products in the 'Electronics' category: 
- Smartphone (E1001): 20
- Laptop (E1002): 15
- Tablet (E1003): 25
Citations:
start=3 end=20 text='28 September 2023' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'
start=26 end=53 text='total sales amount was 5000' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'
start=60 end=7

In [ ]:
for turn in chat_history:
    print(turn, "\n")

role='USER' message="Can you provide a sales summary for 28th and 29th September 2023                     as well as the stock level of the products in the 'Electronics' category?" tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'}), ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), ToolCall(name='product_database', parameters={'category': 'Electronics'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'}), outputs=[{'date': '2023-09-28', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100'}]), ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), outputs=[{'date': '2023-09-29', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250'}]), ToolResult(call=ToolCall(name='product_database', parameters={'category': 'Electronics'}), outputs=[{'category': 'Electronics', 'products': [{'name

## Tools not required

In [14]:
chat_history = run_assistant("Give me 3 concise tips on how to build a great company")


Question:
Give me 3 concise tips on how to build a great company
--------------------------------------------------
Final response:
1. Focus on your customers and their needs
2. Hire the right people and treat them well
3. Be agile and adapt to change


In [ ]:
for turn in chat_history:
    print(turn, "\n")

## Required tools NA

In [16]:
chat_history = run_assistant("How many employees does this company have?")

Question:
How many employees does this company have?
--------------------------------------------------
Final response:
I'm sorry, I don't have access to that information.


In [ ]:
for turn in chat_history:
    print(turn, "\n")

## State management - memory

In [18]:
chat_history = run_assistant("Can you provide a sales summary for 29th September 2023?")

Question:
Can you provide a sales summary for 29th September 2023?
--------------------------------------------------
Tool calls:
Tool name: daily_sales_report | Parameters: {'day': '2023-09-29'}
Final response:
On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.
Citations:
start=30 end=39 text='250 units' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'
start=64 end=70 text='10,000' document_ids=['daily_sales_report:0:2:0'] type='TEXT_CONTENT'

Cited Documents:
{'date': '2023-09-29', 'id': 'daily_sales_report:0:2:0', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250', 'tool_name': 'daily_sales_report'}


In [ ]:
for turn in chat_history:
    print(turn, "\n")

role='USER' message='Can you provide a sales summary for 29th September 2023?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), outputs=[{'date': '2023-09-29', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250'}])] 

role='CHATBOT' message='On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.' tool_calls=None 



In [20]:
chat_history = run_assistant("What about the 28th?", chat_history)

Question:
What about the 28th?
--------------------------------------------------
Tool calls:
Tool name: daily_sales_report | Parameters: {'day': '2023-09-28'}
Final response:
On 28 September 2023, we sold 100 units, totalling a revenue of 5,000.
Citations:
start=30 end=39 text='100 units' document_ids=['daily_sales_report:0:6:0'] type='TEXT_CONTENT'
start=53 end=69 text='revenue of 5,000' document_ids=['daily_sales_report:0:6:0'] type='TEXT_CONTENT'

Cited Documents:
{'date': '2023-09-28', 'id': 'daily_sales_report:0:6:0', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100', 'tool_name': 'daily_sales_report'}


In [ ]:
for turn in chat_history:
    print(turn, "\n")

role='USER' message='Can you provide a sales summary for 29th September 2023?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), outputs=[{'date': '2023-09-29', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250'}])] 

role='CHATBOT' message='On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.' tool_calls=None 

role='USER' message='What about the 28th?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'}), outputs=[{'date': '2023-09-28', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100'}])] 

role='CHATBOT' message='On 28 September 2023, we sold 100 units, totalling a

In [22]:
chat_history = run_assistant("How many units were sold over both days", chat_history)


Question:
How many units were sold over both days
--------------------------------------------------
Tool calls:
Tool name: daily_sales_report | Parameters: {'day': '2023-09-29'}
Tool name: daily_sales_report | Parameters: {'day': '2023-09-28'}
Final response:
Over the two days, we sold 350 units.
Citations:
start=27 end=36 text='350 units' document_ids=['daily_sales_report:0:10:0', 'daily_sales_report:1:10:0'] type='TEXT_CONTENT'

Cited Documents:
{'date': '2023-09-29', 'id': 'daily_sales_report:0:10:0', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250', 'tool_name': 'daily_sales_report'}
{'date': '2023-09-28', 'id': 'daily_sales_report:1:10:0', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100', 'tool_name': 'daily_sales_report'}


In [ ]:
for turn in chat_history:
    print(turn, "\n")

role='USER' message='Can you provide a sales summary for 29th September 2023?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-29'}), outputs=[{'date': '2023-09-29', 'summary': 'Total Sales Amount: 10000, Total Units Sold: 250'}])] 

role='CHATBOT' message='On 29 September 2023, we sold 250 units, totalling a revenue of 10,000.' tool_calls=None 

role='USER' message='What about the 28th?' tool_calls=None 

role='CHATBOT' message=None tool_calls=[ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'})] 

role='TOOL' tool_results=[ToolResult(call=ToolCall(name='daily_sales_report', parameters={'day': '2023-09-28'}), outputs=[{'date': '2023-09-28', 'summary': 'Total Sales Amount: 5000, Total Units Sold: 100'}])] 

role='CHATBOT' message='On 28 September 2023, we sold 100 units, totalling a